In [4]:
# ============================================================
# ELITE HEART DISEASE PIPELINE
# Dataset: BRFSS 2015 Heart Disease Indicators
# Author-ready workflow for portfolio / academic use
# ============================================================

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    roc_curve,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    balanced_accuracy_score
)

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# Optional but recommended
try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except Exception:
    XGB_AVAILABLE = False


# ============================================================
# 1. CONFIG
# ============================================================

DATA_PATH = Path("heart_disease_health_indicators.csv")
OUTPUT_DIR = Path("heart_disease_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

TARGET = "HeartDiseaseorAttack"
RANDOM_STATE = 42
TEST_SIZE = 0.20
USE_SMOTE = True           # recommended for this dataset
TOP_N_FEATURES = [5, 8, 10, 12]
PLOT_DPI = 180


# ============================================================
# 2. LOAD DATA
# ============================================================

df = pd.read_csv(DATA_PATH)

print("\n" + "="*70)
print("DATASET LOADED")
print("="*70)
print(f"Shape: {df.shape}")
print("\nColumns:")
print(df.columns.tolist())

# Basic cleanup: convert float-like binary/integer values to numeric
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Safety check
assert TARGET in df.columns, f"Target column '{TARGET}' not found."

# Drop duplicates if any
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
after = len(df)

print(f"\nDuplicates removed: {before - after}")
print(f"Final shape: {df.shape}")


# ============================================================
# 3. DATA AUDIT
# ============================================================

def dataset_audit(dataframe, target_col):
    audit = {}

    audit["shape"] = dataframe.shape
    audit["missing_values_per_column"] = dataframe.isna().sum().to_dict()
    audit["total_missing_values"] = int(dataframe.isna().sum().sum())
    audit["dtypes"] = dataframe.dtypes.astype(str).to_dict()
    audit["duplicates"] = int(dataframe.duplicated().sum())

    class_counts = dataframe[target_col].value_counts(dropna=False).sort_index()
    class_pct = dataframe[target_col].value_counts(normalize=True, dropna=False).sort_index() * 100

    audit["class_counts"] = {str(k): int(v) for k, v in class_counts.items()}
    audit["class_percentages"] = {str(k): round(float(v), 2) for k, v in class_pct.items()}

    numeric_summary = dataframe.describe().T
    numeric_summary.to_csv(OUTPUT_DIR / "numeric_summary.csv")

    audit_path = OUTPUT_DIR / "dataset_audit.json"
    with open(audit_path, "w") as f:
        json.dump(audit, f, indent=4)

    return audit, numeric_summary

audit, numeric_summary = dataset_audit(df, TARGET)

print("\n" + "="*70)
print("DATA AUDIT")
print("="*70)
print(json.dumps(audit, indent=4))


# ============================================================
# 4. QUICK PREVIEW TABLE
# ============================================================

preview_df = pd.DataFrame({
    "column": df.columns,
    "sample_value": [df[c].iloc[0] for c in df.columns],
    "dtype": [str(df[c].dtype) for c in df.columns],
    "missing_count": [int(df[c].isna().sum()) for c in df.columns]
})
preview_df.to_csv(OUTPUT_DIR / "column_preview.csv", index=False)
print("\nColumn preview saved to:", OUTPUT_DIR / "column_preview.csv")


# ============================================================
# 5. FEATURE SETUP
# ============================================================

X = df.drop(columns=[TARGET]).copy()
y = df[TARGET].astype(int).copy()

# Treat all predictors as numeric
numeric_features = X.columns.tolist()

# BRFSS coding note:
# several variables are ordinal or binary, but for practical modelling
# they can remain numeric here. For logistic regression, scaling helps.

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features)
    ],
    remainder="drop"
)

# For tree models we usually do not need scaling, but keeping a consistent
# preprocessing pipeline is cleaner and reproducible.


# ============================================================
# 6. TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("\n" + "="*70)
print("TRAIN / TEST SPLIT")
print("="*70)
print(f"X_train: {X_train.shape}")
print(f"X_test : {X_test.shape}")
print(f"Train positive rate: {y_train.mean():.4f}")
print(f"Test positive rate : {y_test.mean():.4f}")


# ============================================================
# 7. EDA PLOTS
# ============================================================

def save_class_distribution(y_series):
    counts = y_series.value_counts().sort_index()
    labels = ["No Heart Disease", "Heart Disease"]

    plt.figure(figsize=(7, 5))
    plt.bar(labels, counts.values)
    plt.title("Target Class Distribution")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "class_distribution.png", dpi=PLOT_DPI)
    plt.close()

def save_top_correlations(dataframe, target_col, top_n=15):
    corr = dataframe.corr(numeric_only=True)[target_col].drop(target_col).sort_values(
        key=lambda s: s.abs(), ascending=False
    )
    corr_top = corr.head(top_n)

    plt.figure(figsize=(9, 6))
    plt.barh(corr_top.index[::-1], corr_top.values[::-1])
    plt.title(f"Top {top_n} Correlations with {target_col}")
    plt.xlabel("Correlation")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "top_target_correlations.png", dpi=PLOT_DPI)
    plt.close()

    corr_top.to_csv(OUTPUT_DIR / "top_target_correlations.csv", header=["correlation"])

def save_feature_boxplots(dataframe, target_col, features):
    for feature in features:
        plt.figure(figsize=(7, 5))
        groups = [
            dataframe.loc[dataframe[target_col] == 0, feature].dropna(),
            dataframe.loc[dataframe[target_col] == 1, feature].dropna()
        ]
        plt.boxplot(groups, labels=["No HD", "HD"])
        plt.title(f"{feature} by Heart Disease Class")
        plt.ylabel(feature)
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / f"boxplot_{feature}.png", dpi=PLOT_DPI)
        plt.close()

save_class_distribution(y)
save_top_correlations(df, TARGET, top_n=15)

eda_features = ["BMI", "GenHlth", "PhysHlth", "MentHlth", "Age", "Income", "Education"]
eda_features = [f for f in eda_features if f in df.columns]
save_feature_boxplots(df, TARGET, eda_features)


# ============================================================
# 8. MODEL DEFINITIONS
# ============================================================

def make_logistic_pipeline(use_smote=True):
    steps = [("preprocessor", preprocessor)]
    if use_smote:
        steps.append(("smote", SMOTE(random_state=RANDOM_STATE)))
    steps.append((
        "model",
        LogisticRegression(
            max_iter=2000,
            class_weight=None if use_smote else "balanced",
            random_state=RANDOM_STATE
        )
    ))
    return ImbPipeline(steps)

def make_rf_pipeline(use_smote=True):
    steps = [("preprocessor", preprocessor)]
    if use_smote:
        steps.append(("smote", SMOTE(random_state=RANDOM_STATE)))
    steps.append((
        "model",
        RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            min_samples_split=8,
            min_samples_leaf=3,
            n_jobs=-1,
            class_weight=None if use_smote else "balanced_subsample",
            random_state=RANDOM_STATE
        )
    ))
    return ImbPipeline(steps)

def make_xgb_pipeline(use_smote=True):
    if not XGB_AVAILABLE:
        return None
    steps = [("preprocessor", preprocessor)]
    if use_smote:
        steps.append(("smote", SMOTE(random_state=RANDOM_STATE)))
    steps.append((
        "model",
        XGBClassifier(
            n_estimators=350,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    ))
    return ImbPipeline(steps)

models = {
    "LogisticRegression": make_logistic_pipeline(use_smote=USE_SMOTE),
    "RandomForest": make_rf_pipeline(use_smote=USE_SMOTE),
}

if XGB_AVAILABLE:
    models["XGBoost"] = make_xgb_pipeline(use_smote=USE_SMOTE)


# ============================================================
# 9. CROSS-VALIDATION
# ============================================================

def run_cross_validation(model_dict, X_data, y_data):
    scoring = {
        "accuracy": "accuracy",
        "balanced_accuracy": "balanced_accuracy",
        "precision": "precision",
        "recall": "recall",
        "f1": "f1",
        "roc_auc": "roc_auc",
        "pr_auc": "average_precision"
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    records = []

    for name, model in model_dict.items():
        print(f"\nRunning CV for: {name}")
        scores = cross_validate(
            model, X_data, y_data,
            scoring=scoring,
            cv=cv,
            n_jobs=1,
            return_train_score=False
        )
        row = {"model": name}
        for metric in scoring.keys():
            row[f"{metric}_mean"] = round(np.mean(scores[f"test_{metric}"]), 4)
            row[f"{metric}_std"] = round(np.std(scores[f"test_{metric}"]), 4)
        records.append(row)

    results_df = pd.DataFrame(records).sort_values("roc_auc_mean", ascending=False)
    results_df.to_csv(OUTPUT_DIR / "cross_validation_results.csv", index=False)
    return results_df

cv_results = run_cross_validation(models, X_train, y_train)

print("\n" + "="*70)
print("CROSS-VALIDATION RESULTS")
print("="*70)
print(cv_results)


# ============================================================
# 10. TEST EVALUATION
# ============================================================

def evaluate_model(name, model, X_train, y_train, X_test, y_test):
    print("\n" + "="*70)
    print(f"TEST EVALUATION: {name}")
    print("="*70)

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    metrics = {
        "model": name,
        "accuracy": round(accuracy_score(y_test, y_pred), 4),
        "balanced_accuracy": round(balanced_accuracy_score(y_test, y_pred), 4),
        "precision": round(precision_score(y_test, y_pred, zero_division=0), 4),
        "recall": round(recall_score(y_test, y_pred, zero_division=0), 4),
        "f1": round(f1_score(y_test, y_pred, zero_division=0), 4),
        "roc_auc": round(roc_auc_score(y_test, y_prob), 4),
        "pr_auc": round(average_precision_score(y_test, y_prob), 4)
    }

    print(pd.Series(metrics))

    # Classification report
    report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
    report_df = pd.DataFrame(report).T
    report_df.to_csv(OUTPUT_DIR / f"{name}_classification_report.csv")

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    cm_df = pd.DataFrame(
        cm,
        index=["Actual_0", "Actual_1"],
        columns=["Pred_0", "Pred_1"]
    )
    cm_df.to_csv(OUTPUT_DIR / f"{name}_confusion_matrix.csv")

    # ROC Curve
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    plt.figure(figsize=(7, 5))
    plt.plot(fpr, tpr, label=f"{name} (AUC={metrics['roc_auc']})")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curve - {name}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"{name}_roc_curve.png", dpi=PLOT_DPI)
    plt.close()

    # PR Curve
    precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_prob)
    plt.figure(figsize=(7, 5))
    plt.plot(recall_vals, precision_vals, label=f"{name} (AP={metrics['pr_auc']})")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"Precision-Recall Curve - {name}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"{name}_pr_curve.png", dpi=PLOT_DPI)
    plt.close()

    # Save predictions
    pred_df = X_test.copy()
    pred_df["actual"] = y_test.values
    pred_df["predicted"] = y_pred
    pred_df["probability"] = y_prob
    pred_df.to_csv(OUTPUT_DIR / f"{name}_test_predictions.csv", index=False)

    return metrics, model

test_metrics = []
fitted_models = {}

for model_name, model in models.items():
    metrics, fitted = evaluate_model(model_name, model, X_train, y_train, X_test, y_test)
    test_metrics.append(metrics)
    fitted_models[model_name] = fitted

test_results_df = pd.DataFrame(test_metrics).sort_values("roc_auc", ascending=False)
test_results_df.to_csv(OUTPUT_DIR / "test_results_summary.csv", index=False)

print("\n" + "="*70)
print("TEST RESULTS SUMMARY")
print("="*70)
print(test_results_df)


# ============================================================
# 11. FEATURE IMPORTANCE / INTERPRETATION
# ============================================================

def get_feature_names_from_preprocessor(preprocessor_obj):
    return numeric_features

def extract_feature_importance(fitted_pipeline, model_name):
    feature_names = get_feature_names_from_preprocessor(preprocessor)

    model = fitted_pipeline.named_steps["model"]

    if hasattr(model, "feature_importances_"):
        importance = pd.DataFrame({
            "feature": feature_names,
            "importance": model.feature_importances_
        }).sort_values("importance", ascending=False)

    elif hasattr(model, "coef_"):
        importance = pd.DataFrame({
            "feature": feature_names,
            "importance": np.abs(model.coef_[0])
        }).sort_values("importance", ascending=False)
    else:
        return None

    importance.to_csv(OUTPUT_DIR / f"{model_name}_feature_importance.csv", index=False)

    plt.figure(figsize=(9, 6))
    top = importance.head(15).sort_values("importance", ascending=True)
    plt.barh(top["feature"], top["importance"])
    plt.title(f"Top 15 Feature Importance - {model_name}")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"{model_name}_feature_importance.png", dpi=PLOT_DPI)
    plt.close()

    return importance

importance_tables = {}
for model_name, fitted in fitted_models.items():
    imp = extract_feature_importance(fitted, model_name)
    if imp is not None:
        importance_tables[model_name] = imp

# Choose best model by ROC AUC
best_model_name = test_results_df.iloc[0]["model"]
best_importance = importance_tables.get(best_model_name, None)

print(f"\nBest model based on ROC AUC: {best_model_name}")


# ============================================================
# 12. REDUCED-FEATURE SCREENING MODELS
# ============================================================

def make_reduced_pipeline(feature_subset, model_type="logistic", use_smote=True):
    reduced_preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]), feature_subset)
        ],
        remainder="drop"
    )

    if model_type == "logistic":
        model = LogisticRegression(
            max_iter=2000,
            class_weight=None if use_smote else "balanced",
            random_state=RANDOM_STATE
        )
    elif model_type == "rf":
        model = RandomForestClassifier(
            n_estimators=250,
            min_samples_split=8,
            min_samples_leaf=3,
            n_jobs=-1,
            class_weight=None if use_smote else "balanced_subsample",
            random_state=RANDOM_STATE
        )
    else:
        raise ValueError("model_type must be 'logistic' or 'rf'")

    steps = [("preprocessor", reduced_preprocessor)]
    if use_smote:
        steps.append(("smote", SMOTE(random_state=RANDOM_STATE)))
    steps.append(("model", model))

    return ImbPipeline(steps)

screening_results = []

if best_importance is not None:
    ranked_features = best_importance["feature"].tolist()

    for n in TOP_N_FEATURES:
        feature_subset = ranked_features[:n]

        for model_type in ["logistic", "rf"]:
            name = f"Reduced_{model_type.upper()}_Top{n}"
            pipeline = make_reduced_pipeline(feature_subset, model_type=model_type, use_smote=USE_SMOTE)

            pipeline.fit(X_train[feature_subset], y_train)
            y_pred = pipeline.predict(X_test[feature_subset])
            y_prob = pipeline.predict_proba(X_test[feature_subset])[:, 1]

            row = {
                "model": name,
                "n_features": n,
                "features": ", ".join(feature_subset),
                "accuracy": round(accuracy_score(y_test, y_pred), 4),
                "balanced_accuracy": round(balanced_accuracy_score(y_test, y_pred), 4),
                "precision": round(precision_score(y_test, y_pred, zero_division=0), 4),
                "recall": round(recall_score(y_test, y_pred, zero_division=0), 4),
                "f1": round(f1_score(y_test, y_pred, zero_division=0), 4),
                "roc_auc": round(roc_auc_score(y_test, y_prob), 4),
                "pr_auc": round(average_precision_score(y_test, y_prob), 4),
            }
            screening_results.append(row)

screening_df = pd.DataFrame(screening_results).sort_values(
    ["roc_auc", "recall", "f1"], ascending=False
)
screening_df.to_csv(OUTPUT_DIR / "reduced_feature_screening_results.csv", index=False)

print("\n" + "="*70)
print("REDUCED-FEATURE SCREENING RESULTS")
print("="*70)
print(screening_df.head(10))


# ============================================================
# 13. RISK SEGMENTATION USING BEST MODEL
# ============================================================

best_fitted_model = fitted_models[best_model_name]
best_probs = best_fitted_model.predict_proba(X_test)[:, 1]

risk_df = X_test.copy()
risk_df["actual"] = y_test.values
risk_df["risk_probability"] = best_probs

risk_df["risk_band"] = pd.cut(
    risk_df["risk_probability"],
    bins=[0.0, 0.2, 0.5, 0.8, 1.0],
    labels=["Low", "Moderate", "High", "Very High"],
    include_lowest=True
)

risk_summary = risk_df.groupby("risk_band", observed=False).agg(
    count=("risk_probability", "size"),
    mean_probability=("risk_probability", "mean"),
    observed_case_rate=("actual", "mean")
).reset_index()

risk_summary.to_csv(OUTPUT_DIR / "risk_band_summary.csv", index=False)

plt.figure(figsize=(7, 5))
risk_summary_plot = risk_summary.copy()
plt.bar(risk_summary_plot["risk_band"].astype(str), risk_summary_plot["count"])
plt.title("Test Set Risk Band Distribution")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "risk_band_distribution.png", dpi=PLOT_DPI)
plt.close()


# ============================================================
# 14. FINAL ANALYST SUMMARY
# ============================================================

summary_lines = []
summary_lines.append("HEART DISEASE ELITE PIPELINE SUMMARY")
summary_lines.append("=" * 50)
summary_lines.append(f"Dataset shape: {df.shape}")
summary_lines.append(f"Target positive rate: {y.mean():.4f}")
summary_lines.append(f"Missing values: {int(df.isna().sum().sum())}")
summary_lines.append(f"Best model: {best_model_name}")

best_row = test_results_df.iloc[0]
summary_lines.append(
    f"Best model metrics -> ROC AUC: {best_row['roc_auc']}, "
    f"PR AUC: {best_row['pr_auc']}, Recall: {best_row['recall']}, F1: {best_row['f1']}"
)

if best_importance is not None:
    summary_lines.append("Top 10 important features:")
    for _, row in best_importance.head(10).iterrows():
        summary_lines.append(f" - {row['feature']}: {row['importance']:.4f}")

summary_text = "\n".join(summary_lines)
print("\n" + "="*70)
print(summary_text)
print("="*70)

with open(OUTPUT_DIR / "final_summary.txt", "w", encoding="utf-8") as f:
    f.write(summary_text)

print("\nAll outputs saved to:", OUTPUT_DIR.resolve())


DATASET LOADED
Shape: (253680, 22)

Columns:
['HeartDiseaseorAttack', 'HighBP', 'HighChol', 'CholCheck', 'BMI', 'Smoker', 'Stroke', 'Diabetes', 'PhysActivity', 'Fruits', 'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth', 'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Education', 'Income']

Duplicates removed: 23899
Final shape: (229781, 22)

DATA AUDIT
{
    "shape": [
        229781,
        22
    ],
    "missing_values_per_column": {
        "HeartDiseaseorAttack": 0,
        "HighBP": 0,
        "HighChol": 0,
        "CholCheck": 0,
        "BMI": 0,
        "Smoker": 0,
        "Stroke": 0,
        "Diabetes": 0,
        "PhysActivity": 0,
        "Fruits": 0,
        "Veggies": 0,
        "HvyAlcoholConsump": 0,
        "AnyHealthcare": 0,
        "NoDocbcCost": 0,
        "GenHlth": 0,
        "MentHlth": 0,
        "PhysHlth": 0,
        "DiffWalk": 0,
        "Sex": 0,
        "Age": 0,
        "Education": 0,
        "Income": 0
    },
    "total

In [8]:
from sklearn.metrics import precision_recall_curve, roc_curve, f1_score, precision_score, recall_score, balanced_accuracy_score, accuracy_score
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def optimize_threshold(y_true, y_prob, min_precision=0.25, min_recall=None, save_path=None, model_name="Model"):
    """
    Finds useful classification thresholds for imbalanced binary classification.

    Returns:
        thresholds_df: dataframe of threshold performance
        best_f1_row: threshold with highest F1
        best_precision_constrained_recall_row: highest recall subject to min precision
        best_youden_row: threshold maximizing Youden's J statistic
    """

    # ------------------------------------------------------------
    # 1. Build threshold grid
    # ------------------------------------------------------------
    threshold_grid = np.arange(0.01, 1.00, 0.01)
    rows = []

    for t in threshold_grid:
        y_pred_t = (y_prob >= t).astype(int)

        acc = accuracy_score(y_true, y_pred_t)
        bal_acc = balanced_accuracy_score(y_true, y_pred_t)
        prec = precision_score(y_true, y_pred_t, zero_division=0)
        rec = recall_score(y_true, y_pred_t, zero_division=0)
        f1 = f1_score(y_true, y_pred_t, zero_division=0)

        rows.append({
            "threshold": round(float(t), 2),
            "accuracy": acc,
            "balanced_accuracy": bal_acc,
            "precision": prec,
            "recall": rec,
            "f1": f1
        })

    thresholds_df = pd.DataFrame(rows)

    # ------------------------------------------------------------
    # 2. Best F1 threshold
    # ------------------------------------------------------------
    best_f1_row = thresholds_df.loc[thresholds_df["f1"].idxmax()].copy()

    # ------------------------------------------------------------
    # 3. Best recall with minimum precision constraint
    # ------------------------------------------------------------
    constrained_df = thresholds_df[thresholds_df["precision"] >= min_precision].copy()

    if min_recall is not None:
        constrained_df = constrained_df[constrained_df["recall"] >= min_recall].copy()

    if len(constrained_df) > 0:
        best_precision_constrained_recall_row = constrained_df.sort_values(
            ["recall", "f1", "balanced_accuracy"],
            ascending=False
        ).iloc[0].copy()
    else:
        best_precision_constrained_recall_row = None

    # ------------------------------------------------------------
    # 4. Best Youden J threshold from ROC
    # ------------------------------------------------------------
    fpr, tpr, roc_thresholds = roc_curve(y_true, y_prob)
    youden_j = tpr - fpr
    best_idx = np.argmax(youden_j)

    best_youden_threshold = roc_thresholds[best_idx]
    y_pred_youden = (y_prob >= best_youden_threshold).astype(int)

    best_youden_row = pd.Series({
        "threshold": float(best_youden_threshold),
        "accuracy": accuracy_score(y_true, y_pred_youden),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred_youden),
        "precision": precision_score(y_true, y_pred_youden, zero_division=0),
        "recall": recall_score(y_true, y_pred_youden, zero_division=0),
        "f1": f1_score(y_true, y_pred_youden, zero_division=0)
    })

    # ------------------------------------------------------------
    # 5. Save outputs
    # ------------------------------------------------------------
    if save_path is not None:
        thresholds_df.to_csv(save_path / f"{model_name}_threshold_optimization.csv", index=False)

        # Precision-Recall vs Threshold
        plt.figure(figsize=(8, 5))
        plt.plot(thresholds_df["threshold"], thresholds_df["precision"], label="Precision")
        plt.plot(thresholds_df["threshold"], thresholds_df["recall"], label="Recall")
        plt.plot(thresholds_df["threshold"], thresholds_df["f1"], label="F1")
        plt.xlabel("Threshold")
        plt.ylabel("Score")
        plt.title(f"Threshold Optimization - {model_name}")
        plt.legend()
        plt.tight_layout()
        plt.savefig(save_path / f"{model_name}_threshold_tradeoff.png", dpi=180)
        plt.close()

    return thresholds_df, best_f1_row, best_precision_constrained_recall_row, best_youden_row

In [10]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

In [14]:
# Threshold optimization
thresholds_df, best_f1_row, best_recall_row, best_youden_row = optimize_threshold(
    y_true=y_test,
    y_prob=y_prob,
    min_precision=0.25,
    save_path=OUTPUT_DIR,
    model_name=name
)

print("\nBest threshold by F1:")
print(best_f1_row)

print("\nBest threshold by recall with precision >= 0.25:")
print(best_recall_row if best_recall_row is not None else "No threshold met the constraint.")

print("\nBest threshold by Youden J:")
print(best_youden_row)

# Use recall-priority threshold for screening
if best_recall_row is not None:
    selected_threshold = float(best_recall_row["threshold"])
else:
    selected_threshold = float(best_f1_row["threshold"])

y_pred_adjusted = (y_prob >= selected_threshold).astype(int)

print(f"\nSelected threshold: {selected_threshold:.2f}")
print("\nAdjusted classification report:")
print(classification_report(y_test, y_pred_adjusted, zero_division=0))


Best threshold by F1:
threshold            0.260000
accuracy             0.834933
balanced_accuracy    0.723540
precision            0.330269
recall               0.583175
f1                   0.421711
Name: 25, dtype: float64

Best threshold by recall with precision >= 0.25:
threshold            0.150000
accuracy             0.736123
balanced_accuracy    0.756419
precision            0.250574
recall               0.781995
f1                   0.379534
Name: 14, dtype: float64

Best threshold by Youden J:
threshold            0.124239
accuracy             0.700524
balanced_accuracy    0.757561
precision            0.232946
recall               0.829433
f1                   0.363737
dtype: float64

Selected threshold: 0.15

Adjusted classification report:
              precision    recall  f1-score   support

           0       0.97      0.73      0.83     41214
           1       0.25      0.78      0.38      4743

    accuracy                           0.74     45957
   macro avg    

In [16]:
risk_df["risk_band"] = pd.cut(
    risk_df["risk_probability"],
    bins=[0.0, 0.15, 0.30, 0.60, 1.0],
    labels=["Low", "Moderate", "High", "Critical"]
)

In [18]:
import shap

In [20]:
# ============================================================
# 15. SHAP INTERPRETATION FOR TREE MODELS
# ============================================================

def run_shap_analysis(fitted_pipeline, X_train, X_test, model_name, output_dir, sample_size=5000):
    """
    Run SHAP interpretation for fitted tree-based pipeline models.
    Works best for RandomForest and XGBoost.
    """

    print("\n" + "="*70)
    print(f"SHAP ANALYSIS: {model_name}")
    print("="*70)

    # Extract fitted preprocessor and fitted model
    fitted_preprocessor = fitted_pipeline.named_steps["preprocessor"]
    fitted_model = fitted_pipeline.named_steps["model"]

    # Use original numeric feature names
    feature_names = numeric_features.copy()

    # Transform train/test
    X_train_transformed = fitted_preprocessor.transform(X_train)
    X_test_transformed = fitted_preprocessor.transform(X_test)

    # Convert to DataFrame for readability
    X_train_transformed_df = pd.DataFrame(X_train_transformed, columns=feature_names, index=X_train.index)
    X_test_transformed_df = pd.DataFrame(X_test_transformed, columns=feature_names, index=X_test.index)

    # Optional subsampling for speed
    if len(X_test_transformed_df) > sample_size:
        X_shap = X_test_transformed_df.sample(sample_size, random_state=RANDOM_STATE).copy()
    else:
        X_shap = X_test_transformed_df.copy()

    # TreeExplainer works well for XGBoost / RF
    explainer = shap.TreeExplainer(fitted_model)
    shap_values = explainer.shap_values(X_shap)

    # For binary classification, shap_values may return:
    # - array (n_samples, n_features)
    # - or list of two arrays depending on version/setup
    if isinstance(shap_values, list):
        shap_array = shap_values[1] if len(shap_values) > 1 else shap_values[0]
    else:
        shap_array = shap_values

    # If 3D shape appears, reduce to positive class
    if len(np.shape(shap_array)) == 3:
        shap_array = shap_array[:, :, 1]

    # ------------------------------------------------------------
    # 1. Mean absolute SHAP importance
    # ------------------------------------------------------------
    mean_abs_shap = np.abs(shap_array).mean(axis=0)

    shap_importance_df = pd.DataFrame({
        "feature": X_shap.columns,
        "mean_abs_shap": mean_abs_shap
    }).sort_values("mean_abs_shap", ascending=False)

    shap_importance_df.to_csv(output_dir / f"{model_name}_shap_importance.csv", index=False)

    print("\nTop SHAP drivers:")
    print(shap_importance_df.head(15))

    # ------------------------------------------------------------
    # 2. SHAP summary bar plot
    # ------------------------------------------------------------
    plt.figure()
    shap.summary_plot(
        shap_array,
        X_shap,
        plot_type="bar",
        show=False
    )
    plt.tight_layout()
    plt.savefig(output_dir / f"{model_name}_shap_summary_bar.png", dpi=180, bbox_inches="tight")
    plt.close()

    # ------------------------------------------------------------
    # 3. SHAP summary beeswarm plot
    # ------------------------------------------------------------
    plt.figure()
    shap.summary_plot(
        shap_array,
        X_shap,
        show=False
    )
    plt.tight_layout()
    plt.savefig(output_dir / f"{model_name}_shap_summary_beeswarm.png", dpi=180, bbox_inches="tight")
    plt.close()

    # ------------------------------------------------------------
    # 4. Dependence plots for top features
    # ------------------------------------------------------------
    top_features = shap_importance_df["feature"].head(6).tolist()

    for feat in top_features:
        plt.figure()
        shap.dependence_plot(
            feat,
            shap_array,
            X_shap,
            show=False
        )
        plt.tight_layout()
        safe_feat = str(feat).replace("/", "_")
        plt.savefig(output_dir / f"{model_name}_shap_dependence_{safe_feat}.png", dpi=180, bbox_inches="tight")
        plt.close()

    # ------------------------------------------------------------
    # 5. Direction of effect estimate
    # crude but useful: correlation between feature value and SHAP value
    # ------------------------------------------------------------
    effect_rows = []
    for feat in X_shap.columns:
        try:
            corr = np.corrcoef(X_shap[feat], shap_array[:, X_shap.columns.get_loc(feat)])[0, 1]
        except Exception:
            corr = np.nan

        if pd.isna(corr):
            direction = "Unclear / non-linear"
        elif corr > 0.1:
            direction = "Higher values tend to increase predicted risk"
        elif corr < -0.1:
            direction = "Higher values tend to decrease predicted risk"
        else:
            direction = "Weak or non-linear effect"

        effect_rows.append({
            "feature": feat,
            "correlation_feature_vs_shap": corr,
            "effect_direction": direction
        })

    shap_direction_df = pd.DataFrame(effect_rows)
    shap_direction_df = shap_direction_df.merge(shap_importance_df, on="feature", how="left")
    shap_direction_df = shap_direction_df.sort_values("mean_abs_shap", ascending=False)
    shap_direction_df.to_csv(output_dir / f"{model_name}_shap_direction_table.csv", index=False)

    return shap_importance_df, shap_direction_df, X_shap, shap_array

In [22]:
best_model_name = test_results_df.iloc[0]["model"]
best_importance = importance_tables.get(best_model_name, None)
print(f"\nBest model based on ROC AUC: {best_model_name}")


Best model based on ROC AUC: XGBoost


In [24]:
# Run SHAP only for tree-based best models
if best_model_name in ["RandomForest", "XGBoost"]:
    shap_importance_df, shap_direction_df, X_shap_used, shap_values_used = run_shap_analysis(
        fitted_pipeline=fitted_models[best_model_name],
        X_train=X_train,
        X_test=X_test,
        model_name=best_model_name,
        output_dir=OUTPUT_DIR,
        sample_size=5000
    )
else:
    print(f"Skipping SHAP tree analysis for {best_model_name}.")


SHAP ANALYSIS: XGBoost

Top SHAP drivers:
         feature  mean_abs_shap
18           Age       0.926600
13       GenHlth       0.504681
20        Income       0.332246
0         HighBP       0.317678
17           Sex       0.316989
1       HighChol       0.272928
3            BMI       0.249972
19     Education       0.147135
4         Smoker       0.139917
15      PhysHlth       0.130569
5         Stroke       0.097079
16      DiffWalk       0.069199
14      MentHlth       0.059596
7   PhysActivity       0.049287
6       Diabetes       0.043771


<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

In [26]:
def build_plain_english_shap_summary(shap_direction_df, top_n=10):
    top_df = shap_direction_df.head(top_n).copy()

    explanations = []
    for _, row in top_df.iterrows():
        feat = row["feature"]
        direction = row["effect_direction"]

        explanations.append({
            "feature": feat,
            "importance_rank": len(explanations) + 1,
            "mean_abs_shap": row["mean_abs_shap"],
            "plain_english_interpretation": f"{feat}: {direction}."
        })

    out_df = pd.DataFrame(explanations)
    out_df.to_csv(OUTPUT_DIR / "plain_english_shap_summary.csv", index=False)
    return out_df

if best_model_name in ["RandomForest", "XGBoost"]:
    plain_english_shap_df = build_plain_english_shap_summary(shap_direction_df, top_n=10)
    print("\nPlain-English SHAP summary:")
    print(plain_english_shap_df)


Plain-English SHAP summary:
     feature  importance_rank  mean_abs_shap  \
0        Age                1       0.926600   
1    GenHlth                2       0.504681   
2     Income                3       0.332246   
3     HighBP                4       0.317678   
4        Sex                5       0.316989   
5   HighChol                6       0.272928   
6        BMI                7       0.249972   
7  Education                8       0.147135   
8     Smoker                9       0.139917   
9   PhysHlth               10       0.130569   

                        plain_english_interpretation  
0  Age: Higher values tend to increase predicted ...  
1  GenHlth: Higher values tend to increase predic...  
2  Income: Higher values tend to decrease predict...  
3  HighBP: Higher values tend to increase predict...  
4  Sex: Higher values tend to increase predicted ...  
5  HighChol: Higher values tend to increase predi...  
6  BMI: Higher values tend to increase predicted ...  
7 